# Handling Missing Data (`NaN` values)

In real-world datasets, missing values are extremely common. They can occur due to data entry errors, system glitches, or simply because the information was never collected (e.g., an optional field on a form).

Pandas represents missing or null values as **`NaN`** (which stands for **Not a Number**, inherited from NumPy). Another representation is Python's **`None`** object. Working with datasets that contain missing values can break your mathematical formulas or cause your machine learning models to fail. Therefore, learning how to find and clean missing data is a fundamental skill.

### Plain English Explanation & Real-Life Analogy
Imagine you are a **teacher grading exam sheets**.
* Some students left specific questions blank.
* You have three choices on how to handle those empty cells:
  1. **Identify**: Circle the blank spots so you know who skipped questions.
  2. **Drop**: Throw away the entire exam sheet if it's mostly blank (though this is a very harsh decision).
  3. **Fill**: Write a default score (like a zero or the class average) in the blank spaces so you can still run your grading formulas.

This is exactly how Pandas treats missing data.

### Code Examples

Let's start by creating a DataFrame representing a messy coffee sales dataset that contains missing values (`NaN`). We will use `import numpy as np` to create our `NaN` values.


In [5]:
import pandas as pd
import numpy as np

data = {
    'Day': ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday'],
    'Coffee_Type': ['Latte', 'Espresso', np.nan, 'Latte', 'Espresso'],
    'Units_Sold': [25.0, np.nan, np.nan, 30.0, 15.0],
    'Price_USD': [4.99, 3.99, np.nan, 4.99, 3.99]
}

df = pd.DataFrame(data)
print(df)

         Day Coffee_Type  Units_Sold  Price_USD
0     Monday       Latte        25.0       4.99
1    Tuesday    Espresso         NaN       3.99
2  Wednesday         NaN         NaN        NaN
3   Thursday       Latte        30.0       4.99
4     Friday    Espresso        15.0       3.99



#### Finding Missing Values (`.isna()` / `.isnull()` and `.sum()`)
To find missing values, you can use `.isna()` or `.isnull()` (they are complete synonyms; you can use whichever you prefer).

When run on a DataFrame, it returns `True` for every empty cell and `False` for filled cells. Combining this with `.sum()` will count how many missing values are in each column.


In [6]:
# Check which cells are null
print("--- Check Null Cells (.isna) ---")
print(df.isna())

# Count missing values per column
print("--- Missing Values Count per Column ---")
print(df.isna().sum())

--- Check Null Cells (.isna) ---
     Day  Coffee_Type  Units_Sold  Price_USD
0  False        False       False      False
1  False        False        True      False
2  False         True        True       True
3  False        False       False      False
4  False        False       False      False
--- Missing Values Count per Column ---
Day            0
Coffee_Type    1
Units_Sold     2
Price_USD      1
dtype: int64


#### B) Dropping Missing Values (`.dropna()`)
If you decide that any row containing an empty cell should be removed, you can use `.dropna()`.

* **Default Behavior**: Drops any row that has at least one missing value.
* **Using `subset`**: Only drops rows if a missing value is in a *specific* column (e.g., only drop if `Units_Sold` is missing).
* **Memory Note**: Like most Pandas operations, `.dropna()` is **immutable** (creates a copy of the data). To make the changes permanent, you must use `inplace=True` or reassign the DataFrame.


In [7]:
# Drop any row containing at least one NaN (Default)
df_clean_rows = df.dropna()
print("--- Default Drop NA (Harsh) ---")
print(df_clean_rows)

# Drop rows ONLY if they have NaN in the 'Units_Sold' column
df_clean_subset = df.dropna(subset=['Units_Sold'])
print("--- Drop NA using subset on 'Units_Sold' ---")
print(df_clean_subset)

--- Default Drop NA (Harsh) ---
        Day Coffee_Type  Units_Sold  Price_USD
0    Monday       Latte        25.0       4.99
3  Thursday       Latte        30.0       4.99
4    Friday    Espresso        15.0       3.99
--- Drop NA using subset on 'Units_Sold' ---
        Day Coffee_Type  Units_Sold  Price_USD
0    Monday       Latte        25.0       4.99
3  Thursday       Latte        30.0       4.99
4    Friday    Espresso        15.0       3.99


#### Filling Missing Values (`.fillna()`)
Throwing away data is often a bad idea because you lose other valuable information in that row. A smarter approach is to fill empty cells using `.fillna()`.

You can fill missing values with:
1. **A static fallback value** (e.g., filling category gaps with "Unknown" or `0`).
2. **A statistical calculation** (e.g., filling numerical gaps with the column's **mean** or **median**).
3. **A dictionary**: Specifying different fill values for different columns.


In [9]:
# Create a copy to work with
df_filled = df.copy()

# 1. Fill Coffee_Type missing values with a static string "Unknown"
df_filled['Coffee_Type'] = df_filled['Coffee_Type'].fillna("Unknown")

# 2. Fill Units_Sold missing values with the mean of that column
mean_units = df_filled['Units_Sold'].mean()
print(f"Mean Units Sold: {mean_units}")
df_filled['Units_Sold'] = df_filled['Units_Sold'].fillna(mean_units)

# 3. Fill Price_USD using a dictionary-based fillna
df_filled = df_filled.fillna({'Price_USD': 4.50})

print("--- Cleaned & Filled DataFrame ---")
print(df_filled)

Mean Units Sold: 23.333333333333332
--- Cleaned & Filled DataFrame ---
         Day Coffee_Type  Units_Sold  Price_USD
0     Monday       Latte   25.000000       4.99
1    Tuesday    Espresso   23.333333       3.99
2  Wednesday     Unknown   23.333333       4.50
3   Thursday       Latte   30.000000       4.99
4     Friday    Espresso   15.000000       3.99


### Common Pitfalls
1. **Forgetting `inplace=True` or Reassignment**: Writing `df.dropna()` or `df.fillna(0)` and wondering why your original DataFrame still has missing values. Always write `df = df.dropna()` or use `inplace=True`.
2. **Filling Categorical Data with the Mean**: Trying to calculate the average of a text column (like `Coffee_Type`) will result in an error. For text, you should fill with a static string (like `"None"` or `"Unknown"`) or use the column's **mode** (the most frequent value).

#### Exercise 1 (Easy)
You are given a Series of student scores: `scores = pd.Series([85, np.nan, 90, 75, np.nan])`.
1. Find the total number of missing scores.
2. Calculate the mean score and fill the missing scores with that mean.


In [11]:
import pandas as pd
import numpy as np

scores = pd.Series([85, np.nan, 90, 75, np.nan])

# Count missing values
missing_count = scores.isna().sum()
print(f"Number of missing scores: {missing_count}")

# Fill with mean
mean_score = scores.mean()
scores_clean = scores.fillna(mean_score)
print("Cleaned Scores:")
print(scores_clean)

Number of missing scores: 2
Cleaned Scores:
0    85.000000
1    83.333333
2    90.000000
3    75.000000
4    83.333333
dtype: float64



#### Exercise 2 (Medium)
You have a DataFrame of laptop inventory:
```python
inventory = pd.DataFrame({
    'Brand': ['Dell', 'Apple', np.nan, 'HP', 'Apple'],
    'Price_USD': [1200, np.nan, 800, np.nan, 1500]
})
```
Write Pandas code to:
1. Drop any rows where the `Brand` is missing.
2. Fill any remaining missing prices with the median price of the available laptops.

In [12]:
import pandas as pd
import numpy as np

inventory = pd.DataFrame({
    'Brand': ['Dell', 'Apple', np.nan, 'HP', 'Apple'],
    'Price_USD': [1200, np.nan, 800, np.nan, 1500]
})

# 1. Drop rows where Brand is missing
inventory_clean = inventory.dropna(subset=['Brand'])

# 2. Calculate the median of the Price column
median_price = inventory_clean['Price_USD'].median()

# 3. Fill missing prices with the calculated median
inventory_clean['Price_USD'] = inventory_clean['Price_USD'].fillna(median_price)

print(inventory_clean)

   Brand  Price_USD
0   Dell     1200.0
1  Apple     1350.0
3     HP     1350.0
4  Apple     1500.0


*Explanation*: The row at index `2` had a missing brand (`NaN`), so it was successfully dropped. The remaining valid prices were `1200.0` and `1500.0`, whose median is `1350.0`. The missing prices at index `1` and `3` were filled with `1350.0`.
